In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [2]:
base_path = "/kaggle/input/datasets/subhajournal/busi-breast-ultrasound-images-dataset"

image_paths = []
labels = []

for root, dirs, files in os.walk(base_path):

    for file in files:

        if file.endswith(".png") and "mask" not in file:

            path = os.path.join(root,file)

            if "benign" in root.lower():
                label = "benign"

            elif "malignant" in root.lower():
                label = "malignant"

            elif "normal" in root.lower():
                label = "normal"

            else:
                continue

            image_paths.append(path)
            labels.append(label)

df = pd.DataFrame({
    "image": image_paths,
    "label": labels
})

print("Total Images:",len(df))
print("\nOriginal Class Distribution:")
print(df['label'].value_counts())

Total Images: 780

Original Class Distribution:
label
benign       437
malignant    210
normal       133
Name: count, dtype: int64


In [3]:
label_map = {
    "normal":0,
    "benign":1,
    "malignant":2
}

df['label'] = df['label'].map(label_map)

In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df['label'],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label'],
    random_state=42
)

print("Train size:",len(train_df))
print("Validation size:",len(val_df))
print("Test size:",len(test_df))

print("\nTrain Class Distribution:")
print(train_df['label'].value_counts())

Train size: 563
Validation size: 100
Test size: 117

Train Class Distribution:
label
1    315
2    152
0     96
Name: count, dtype: int64


In [5]:
class BUSIDataset(Dataset):

    def __init__(self,dataframe,transform=None):

        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):

        img_path = self.df.iloc[idx]['image']
        label = self.df.iloc[idx]['label']

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image,label

In [6]:
basic_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
aug_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(10,translate=(0.1,0.1)),
    transforms.ToTensor()
])

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:",device)

Using device: cuda


In [8]:
train_dataset = BUSIDataset(train_df,basic_transform)
val_dataset = BUSIDataset(val_df,basic_transform)
test_dataset = BUSIDataset(test_df,basic_transform)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32)
test_loader = DataLoader(test_dataset,batch_size=32)

In [17]:
def train_model(train_loader,val_loader,epochs=15):

    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features,3)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images,labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print("\nEpoch",epoch+1,"Train Loss:",running_loss/len(train_loader))

        # validation
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images,labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _,predicted = torch.max(outputs,1)

                total += labels.size(0)
                correct += (predicted==labels).sum().item()

        print("Validation Accuracy:",correct/total)

    return model

In [18]:
def evaluate_model(model,test_loader):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for images,labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            _,predicted = torch.max(outputs,1)

            y_true.extend(labels.numpy())
            y_pred.extend(predicted.cpu().numpy())

    print(classification_report(
        y_true,
        y_pred,
        target_names=["Normal","Benign","Malignant"]
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true,y_pred))

    return y_true,y_pred

In [19]:
print("\n====================")
print("BASELINE TRAINING")
print("====================")

baseline_model = train_model(train_loader,val_loader)

print("\nBaseline Evaluation")

baseline_true,baseline_pred = evaluate_model(baseline_model,test_loader)


BASELINE TRAINING


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1 Train Loss: 0.7034397191471524
Validation Accuracy: 0.47

Epoch 2 Train Loss: 0.16917639929387304
Validation Accuracy: 0.81

Epoch 3 Train Loss: 0.03993290078101887
Validation Accuracy: 0.89

Epoch 4 Train Loss: 0.03397922693855233
Validation Accuracy: 0.81

Epoch 5 Train Loss: 0.015452240826562047
Validation Accuracy: 0.84

Epoch 6 Train Loss: 0.011121787493013673
Validation Accuracy: 0.87

Epoch 7 Train Loss: 0.008568568245714737
Validation Accuracy: 0.89

Epoch 8 Train Loss: 0.007009778184712761
Validation Accuracy: 0.87

Epoch 9 Train Loss: 0.00875391518153871
Validation Accuracy: 0.89

Epoch 10 Train Loss: 0.010191731601177404
Validation Accuracy: 0.87

Epoch 11 Train Loss: 0.005891780808774961
Validation Accuracy: 0.86

Epoch 12 Train Loss: 0.014418602956640016
Validation Accuracy: 0.86

Epoch 13 Train Loss: 0.007942964436046572
Validation Accuracy: 0.85

Epoch 14 Train Loss: 0.005876230037150283
Validation Accuracy: 0.87

Epoch 15 Train Loss: 0.005221562721999362
Valida

In [20]:
print("\n====================")
print("OVERSAMPLING")
print("====================")

class_counts = train_df['label'].value_counts().sort_index()

print("Original Train Counts:")
print(class_counts)

weights = 1.0 / class_counts

sample_weights = train_df['label'].map(weights)

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_over = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

print("\nTraining with Oversampling")

over_model = train_model(train_loader_over,val_loader)

print("\nOversampling Evaluation")

over_true,over_pred = evaluate_model(over_model,test_loader)


OVERSAMPLING
Original Train Counts:
label
0     96
1    315
2    152
Name: count, dtype: int64

Training with Oversampling


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1 Train Loss: 0.7872145192490684
Validation Accuracy: 0.56

Epoch 2 Train Loss: 0.21874820565183958
Validation Accuracy: 0.69

Epoch 3 Train Loss: 0.10069889099233681
Validation Accuracy: 0.78

Epoch 4 Train Loss: 0.0392357652179069
Validation Accuracy: 0.82

Epoch 5 Train Loss: 0.029041753254002996
Validation Accuracy: 0.84

Epoch 6 Train Loss: 0.013796073151752353
Validation Accuracy: 0.86

Epoch 7 Train Loss: 0.01061856167183982
Validation Accuracy: 0.9

Epoch 8 Train Loss: 0.00735673426081323
Validation Accuracy: 0.88

Epoch 9 Train Loss: 0.007509506928424041
Validation Accuracy: 0.85

Epoch 10 Train Loss: 0.009715816432920596
Validation Accuracy: 0.87

Epoch 11 Train Loss: 0.006083998971411752
Validation Accuracy: 0.85

Epoch 12 Train Loss: 0.0021745708331258762
Validation Accuracy: 0.86

Epoch 13 Train Loss: 0.004790800035051588
Validation Accuracy: 0.87

Epoch 14 Train Loss: 0.00569614540048254
Validation Accuracy: 0.89

Epoch 15 Train Loss: 0.008815777520390434
Validatio

In [21]:
print("\n====================")
print("AUGMENTATION")
print("====================")

train_dataset_aug = BUSIDataset(train_df,aug_transform)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=32,
    shuffle=True
)

print("Using Data Augmentation")

aug_model = train_model(train_loader_aug,val_loader)

print("\nAugmentation Evaluation")

aug_true,aug_pred = evaluate_model(aug_model,test_loader)


AUGMENTATION
Using Data Augmentation


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1 Train Loss: 0.7562298940287696
Validation Accuracy: 0.72

Epoch 2 Train Loss: 0.49340619643529254
Validation Accuracy: 0.78

Epoch 3 Train Loss: 0.3362103088034524
Validation Accuracy: 0.77

Epoch 4 Train Loss: 0.26989122273193467
Validation Accuracy: 0.88

Epoch 5 Train Loss: 0.2081773980624146
Validation Accuracy: 0.86

Epoch 6 Train Loss: 0.18857649051480824
Validation Accuracy: 0.82

Epoch 7 Train Loss: 0.1277179192337725
Validation Accuracy: 0.85

Epoch 8 Train Loss: 0.1091436862738596
Validation Accuracy: 0.88

Epoch 9 Train Loss: 0.11757573464678393
Validation Accuracy: 0.88

Epoch 10 Train Loss: 0.09832800986866157
Validation Accuracy: 0.89

Epoch 11 Train Loss: 0.07876948732882738
Validation Accuracy: 0.86

Epoch 12 Train Loss: 0.08982282690703869
Validation Accuracy: 0.88

Epoch 13 Train Loss: 0.0835726833384898
Validation Accuracy: 0.88

Epoch 14 Train Loss: 0.07563402679645354
Validation Accuracy: 0.87

Epoch 15 Train Loss: 0.05029856931004259
Validation Accuracy: 

In [25]:
class FocalLoss(nn.Module):

    def __init__(self,gamma=2):
        super(FocalLoss,self).__init__()

        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self,inputs,targets):

        ce_loss = self.ce(inputs,targets)

        pt = torch.exp(-ce_loss)

        loss = (1-pt)**self.gamma * ce_loss

        return loss.mean()

In [26]:
def train_focal(train_loader,val_loader,epochs=15):

    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features,3)

    model = model.to(device)

    criterion = FocalLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images,labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print("\nEpoch",epoch+1,"Train Loss:",running_loss/len(train_loader))

        # validation
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images,labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _,predicted = torch.max(outputs,1)

                total += labels.size(0)
                correct += (predicted==labels).sum().item()

        print("Validation Accuracy:",correct/total)

    return model


print("\n====================")
print("FOCAL LOSS TRAINING")
print("====================")

focal_model = train_focal(train_loader,val_loader)

print("\nFocal Loss Evaluation")

focal_true,focal_pred = evaluate_model(focal_model,test_loader)


FOCAL LOSS TRAINING


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1 Train Loss: 0.48610736015770173
Validation Accuracy: 0.56

Epoch 2 Train Loss: 0.17855934922893843
Validation Accuracy: 0.69

Epoch 3 Train Loss: 0.11083084324167834
Validation Accuracy: 0.79

Epoch 4 Train Loss: 0.0727864337257213
Validation Accuracy: 0.73

Epoch 5 Train Loss: 0.052810275409784585
Validation Accuracy: 0.83

Epoch 6 Train Loss: 0.042241297765738435
Validation Accuracy: 0.73

Epoch 7 Train Loss: 0.041315431240946054
Validation Accuracy: 0.78

Epoch 8 Train Loss: 0.04321743092603154
Validation Accuracy: 0.58

Epoch 9 Train Loss: 0.03207676388168087
Validation Accuracy: 0.78

Epoch 10 Train Loss: 0.02046875916292063
Validation Accuracy: 0.85

Epoch 11 Train Loss: 0.010570573656069528
Validation Accuracy: 0.78

Epoch 12 Train Loss: 0.017892400156900596
Validation Accuracy: 0.77

Epoch 13 Train Loss: 0.06946035959602644
Validation Accuracy: 0.83

Epoch 14 Train Loss: 0.03984742984175682
Validation Accuracy: 0.84

Epoch 15 Train Loss: 0.035695424820813865
Validation

In [27]:
results = pd.DataFrame({

    "Method":[
        "Baseline",
        "Oversampling",
        "Augmentation",
        "Focal Loss"
    ],

    "Accuracy":[
        accuracy_score(baseline_true,baseline_pred),
        accuracy_score(over_true,over_pred),
        accuracy_score(aug_true,aug_pred),
        accuracy_score(focal_true,focal_pred)
    ],

    "F1 Score":[
        f1_score(baseline_true,baseline_pred,average='weighted'),
        f1_score(over_true,over_pred,average='weighted'),
        f1_score(aug_true,aug_pred,average='weighted'),
        f1_score(focal_true,focal_pred,average='weighted')
    ]

})

print("\n====================")
print("FINAL COMPARISON")
print("====================")

print(results)


FINAL COMPARISON
         Method  Accuracy  F1 Score
0      Baseline  0.863248  0.853466
1  Oversampling  0.854701  0.842426
2  Augmentation  0.880342  0.876538
3    Focal Loss  0.794872  0.779904
